In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
data_path = '/content/house_price_data/Housing.csv'

try:
    df_house = pd.read_csv(data_path)
    print(f"Successfully loaded {data_path}")
    display(df_house.head())
    print("\nDataFrame Info:")
    df_house.info()
except FileNotFoundError:
    print(f"Error: {data_path} not found. Please check the extracted files.")
    print("Contents of the extracted directory:")
    import os
    for dirname, _, filenames in os.walk('/content/house_price_data'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

Successfully loaded /content/house_price_data/Housing.csv


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [ ]:
numerical_cols = ['price', 'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms', 'floors', 'grade', 'yr_built', 'yr_renovated']

numerical_cols = [col for col in numerical_cols if col in df_house.columns]

df_to_normalize = df_house[numerical_cols].copy()

df_to_normalize.dropna(inplace=True)

if df_to_normalize.empty:
    print("No valid numerical data available for normalization after dropping NaNs.")
else:
    print("Original numerical data for normalization:")
    display(df_to_normalize.head())

    min_max_scaler = MinMaxScaler()
    df_min_max_scaled = pd.DataFrame(
        min_max_scaler.fit_transform(df_to_normalize),
        columns=[f"{col}_MinMax" for col in df_to_normalize.columns]
    )
    print("\nMin-Max Scaled Data (first 5 rows):")
    display(df_min_max_scaled.head())

    z_score_scaler = StandardScaler()
    df_z_score_scaled = pd.DataFrame(
        z_score_scaler.fit_transform(df_to_normalize),
        columns=[f"{col}_ZScore" for col in df_to_normalize.columns]
    )
    print("\nZ-score Scaled Data (first 5 rows):")
    display(df_z_score_scaled.head())

    def decimal_scaling(column):
        max_val = column.abs().max()
        if max_val == 0:
            return column
        j = np.ceil(np.log10(max_val)) if max_val > 0 else 0
        return column / (10**j)

    df_decimal_scaled = df_to_normalize.apply(decimal_scaling).add_suffix("_Decimal")
    print("\nDecimal Scaled Data (first 5 rows):")
    display(df_decimal_scaled.head())

    final_house_df = pd.concat([df_to_normalize.reset_index(drop=True), df_min_max_scaled, df_z_score_scaled, df_decimal_scaled], axis=1)
    print("\nCombined Original and Normalized Data (first 5 rows):")
    display(final_house_df.head())
    print("\nCombined Normalized Data Info:")
    final_house_df.info()

Original numerical data for normalization:


,price,bedrooms,bathrooms
0,13300000,4,2
1,12250000,4,4
2,12250000,3,2
3,12215000,4,2
4,11410000,4,1



Min-Max Scaled Data (first 5 rows):


,price_MinMax,bedrooms_MinMax,bathrooms_MinMax
0,1.000000,0.6,0.333333
1,0.909091,0.6,1.000000
2,0.909091,0.4,0.333333
3,0.906061,0.6,0.333333
4,0.836364,0.6,0.000000



Z-score Scaled Data (first 5 rows):


,price_ZScore,bedrooms_ZScore,bathrooms_ZScore
0,4.566365,1.403419,1.421812
1,4.004484,1.403419,5.405809
2,4.004484,0.047278,1.421812
3,3.985755,1.403419,1.421812
4,3.554979,1.403419,-0.570187



Decimal Scaled Data (first 5 rows):


,price_Decimal,bedrooms_Decimal,bathrooms_Decimal
0,0.13300,0.4,0.2
1,0.12250,0.4,0.4
2,0.12250,0.3,0.2
3,0.12215,0.4,0.2
4,0.11410,0.4,0.1



Combined Original and Normalized Data (first 5 rows):


,price,bedrooms,bathrooms,price_MinMax,bedrooms_MinMax,bathrooms_MinMax,price_ZScore,bedrooms_ZScore,bathrooms_ZScore,price_Decimal,bedrooms_Decimal,bathrooms_Decimal
0,13300000,4,2,1.000000,0.6,0.333333,4.566365,1.403419,1.421812,0.13300,0.4,0.2
1,12250000,4,4,0.909091,0.6,1.000000,4.004484,1.403419,5.405809,0.12250,0.4,0.4
2,12250000,3,2,0.909091,0.4,0.333333,4.004484,0.047278,1.421812,0.12250,0.3,0.2
3,12215000,4,2,0.906061,0.6,0.333333,3.985755,1.403419,1.421812,0.12215,0.4,0.2
4,11410000,4,1,0.836364,0.6,0.000000,3.554979,1.403419,-0.570187,0.11410,0.4,0.1



Combined Normalized Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              545 non-null    int64  
 1   bedrooms           545 non-null    int64  
 2   bathrooms          545 non-null    int64  
 3   price_MinMax       545 non-null    float64
 4   bedrooms_MinMax    545 non-null    float64
 5   bathrooms_MinMax   545 non-null    float64
 6   price_ZScore       545 non-null    float64
 7   bedrooms_ZScore    545 non-null    float64
 8   bathrooms_ZScore   545 non-null    float64
 9   price_Decimal      545 non-null    float64
 10  bedrooms_Decimal   545 non-null    float64
 11  bathrooms_Decimal  545 non-null    float64
dtypes: float64(9), int64(3)
memory usage: 51.2 KB


In [ ]:
from scipy.stats import chi2_contingency

categorical_cols = df_house.select_dtypes(include='object').columns

print("Categorical columns identified:", categorical_cols.tolist())

if len(categorical_cols) >= 2:
    col1 = 'mainroad'
    col2 = 'furnishingstatus'

    if col1 in categorical_cols and col2 in categorical_cols:
        contingency_table = pd.crosstab(df_house[col1], df_house[col2])
        print(f"\nContingency Table for '{col1}' and '{col2}':")
        display(contingency_table)

        chi2, p_value, dof, expected = chi2_contingency(contingency_table)

        print(f"\nChi-square Statistic: {chi2:.4f}")
        print(f"P-value: {p_value:.4f}")
        print(f"Degrees of Freedom: {dof}")

        alpha = 0.05
        if p_value < alpha:
            print(f"\nInterpretation: Since the p-value ({p_value:.4f}) is less than the significance level ({alpha}), we reject the null hypothesis. There is a statistically significant relationship (dependence) between '{col1}' and '{col2}'.")
        else:
            print(f"\nInterpretation: Since the p-value ({p_value:.4f}) is greater than the significance level ({alpha}), we fail to reject the null hypothesis. There is no statistically significant relationship (independence) between '{col1}' and '{col2}'.")
    else:
        print(f"The selected columns '{col1}' or '{col2}' were not found among the categorical columns.")
else:
    print("Not enough categorical columns in the DataFrame to perform a Chi-square test of independence.")

Categorical columns identified: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']

Contingency Table for 'mainroad' and 'furnishingstatus':


furnishingstatus,furnished,semi-furnished,unfurnished
mainroad,,,
no,9,31,37
yes,131,196,141



Chi-square Statistic: 13.3871
P-value: 0.0012
Degrees of Freedom: 2

Interpretation: Since the p-value (0.0012) is less than the significance level (0.05), we reject the null hypothesis. There is a statistically significant relationship (dependence) between 'mainroad' and 'furnishingstatus'.
